# PedVLM-Flow: Enhancing Pedestrian Crossing Intention with Dual-Stream Vision and Optical Flow

This notebook implements the PedVLM-Flow pipeline:
- Correct JAAD annotation parsing
- Motion-enhanced labeling
- Targeted augmentation for minority class
- Class balancing and focal loss
- Dual-stream vision (CLIP + flow CNN) + T5 with temporal attention
- Comprehensive evaluation with visualizations

## Instructions
1) Set `BASE_PATH` in the last cell to your dataset root.
2) Run cells in order.
3) Use `main_enhanced_pipeline()` to run the full pipeline.

## Expected dataset structure:
```
BASE_PATH/
  train/
    videos/*.mp4
    annotations/*.xml
  test/
    videos/*.mp4
    annotations/*.xml
```

## 1. Environment Setup

In [ ]:
# Optional: Install dependencies (uncomment if needed)
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip install transformers==4.41.0 albumentations==1.4.4 opencv-python==4.9.0.80 seaborn==0.13.2 scikit-learn==1.5.0
# !pip install pillow==10.3.0 tqdm==4.66.4 nltk==3.8.1

In [ ]:
import subprocess
import sys

def install_sentencepiece():
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'sentencepiece'])
        print("Successfully installed sentencepiece")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'protobuf'])
        print("Successfully installed protobuf")
        print("Please restart your kernel after installation!")
    except subprocess.CalledProcessError as e:
        print(f"Installation failed: {e}")
        print("Try: pip install sentencepiece protobuf")

install_sentencepiece()

In [ ]:
import os, cv2, random, warnings, math, csv, re
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.sampler import WeightedRandomSampler
from sklearn.metrics import f1_score, classification_report, confusion_matrix, precision_recall_curve, roc_auc_score
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import CLIPProcessor, CLIPVisionModel, T5Tokenizer, T5EncoderModel
from PIL import Image, ImageEnhance, ImageFilter
import xml.etree.ElementTree as ET
from collections import Counter
import nltk
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight
import albumentations as A
from albumentations.pytorch import ToTensorV2

nltk.download('punkt', quiet=True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
def setup_environment_jupyter(base_path):
    paths = {
        "train_videos": os.path.join(base_path, 'train/videos'),
        "test_videos": os.path.join(base_path, 'test/videos'),
        "train_annotations": os.path.join(base_path, 'train/annotations'),
        "test_annotations": os.path.join(base_path, 'test/annotations'),
        "clip_dir": os.path.join(base_path, 'output_clips'),
        "checkpoint_dir": os.path.join(base_path, 'model_checkpoints'),
        "pred_video_dir": os.path.join(base_path, 'pred_videos'),
        "aug_clip_dir": os.path.join(base_path, 'augmented_clips')
    }

    for key in ["clip_dir", "checkpoint_dir", "pred_video_dir", "aug_clip_dir"]:
        os.makedirs(paths[key], exist_ok=True)

    def safe_listdir(p):
        return sorted([f for f in os.listdir(p) if f.endswith('.mp4')]) if os.path.isdir(p) else []

    train_files = [os.path.join(paths["train_videos"], f) for f in safe_listdir(paths["train_videos"])]
    test_files  = [os.path.join(paths["test_videos"], f) for f in safe_listdir(paths["test_videos"])]

    paths["train_files"] = train_files
    paths["test_files"]  = test_files
    print(f"Found {len(train_files)} train videos, {len(test_files)} test videos.")
    return paths

## 2. JAAD Annotation Parsing & Debugging

In [ ]:
def parse_jaad_annotations(root, frame_range):
    try:
        max_frame = max(frame_range)
        min_frame = min(frame_range)
    except ValueError:
        return "not_crossing", 0

    crossing_frames = 0
    total_valid_frames = 0

    for track in root.findall(".//track"):
        track_label = track.attrib.get("label", "")
        if track_label not in ["ped", "pedestrian", "people"]:
            continue

        for box in track.findall("box"):
            try:
                frame_num = int(box.attrib.get("frame", -1))
            except (ValueError, TypeError):
                continue
            if frame_num < min_frame or frame_num > max_frame:
                continue

            total_valid_frames += 1
            for attr in box.findall("attribute"):
                attr_name = attr.attrib.get("name", "")
                attr_text = (attr.text or "")
                if attr_name == "cross" and attr_text.lower() == "crossing":
                    crossing_frames += 1
                    break

    if total_valid_frames == 0:
        return "not_crossing", 0

    crossing_ratio = crossing_frames / total_valid_frames
    return ("crossing", 1) if crossing_ratio > 0.3 else ("not_crossing", 0)


def debug_annotation_content(xml_path, max_frames_to_show=10):
    try:
        root = ET.parse(xml_path).getroot()
        print(f"Debugging annotation: {os.path.basename(xml_path)}")
        tracks = root.findall(".//track")
        print(f"Found {len(tracks)} tracks")
        crossing_frames_found = []
        for i, track in enumerate(tracks):
            label = track.attrib.get("label", "unknown")
            boxes = track.findall("box")
            print(f"  Track {i}: label='{label}', {len(boxes)} boxes")
            track_crossing_frames = 0
            for box in boxes:
                frame = box.attrib.get("frame", "?")
                for attr in box.findall("attribute"):
                    if attr.attrib.get("name") == "cross" and (attr.text or "") == "crossing":
                        track_crossing_frames += 1
                        crossing_frames_found.append(frame)
            if track_crossing_frames > 0:
                print(f"    CROSSING DETECTED: {track_crossing_frames} frames")
            frames_shown = 0
            for box in boxes:
                if frames_shown >= max_frames_to_show:
                    break
                frame = box.attrib.get("frame", "?")
                attrs = box.findall("attribute")
                if attrs:
                    print(f"    Frame {frame}:")
                    for attr in attrs:
                        name = attr.attrib.get("name", "")
                        value = attr.text or ""
                        marker = " <<<< CROSSING!" if (name == "cross" and value == "crossing") else ""
                        print(f"      {name}: {value}{marker}")
                    frames_shown += 1

        print(f"Crossing frames found: {len(crossing_frames_found)}")
        return len(crossing_frames_found) > 0
    except Exception as e:
        print(f"Error parsing {xml_path}: {e}")
        return False


def analyze_all_annotations(annotation_dir):
    if not os.path.exists(annotation_dir):
        print(f"Annotation directory not found: {annotation_dir}")
        return
    xml_files = [f for f in os.listdir(annotation_dir) if f.endswith('.xml')]
    print(f"Found {len(xml_files)} annotation files in {annotation_dir}")
    crossing_videos = []
    for xml_file in xml_files[:5]:
        xml_path = os.path.join(annotation_dir, xml_file)
        print("\n" + "="*80)
        has_crossing = debug_annotation_content(xml_path)
        if has_crossing:
            crossing_videos.append(xml_file)
    print("\n" + "="*80)
    print(f"SUMMARY: {len(crossing_videos)} out of {len(xml_files[:5])} videos have crossing behavior")
    return crossing_videos

## 3. Clip Extraction with Optical Flow

In [ ]:
def extract_clips(video_path, output_dir, num_frames=30, resize=(224,224), stride=10):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        warnings.warn(f"[WARN] Cannot open video {video_path}. Skipping.")
        return []

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame is not None and frame.size > 0:
            resized_frame = cv2.resize(frame, resize)
            frames.append(resized_frame)
    cap.release()

    vid_name = os.path.splitext(os.path.basename(video_path))[0]
    if len(frames) < num_frames:
        warnings.warn(f"[WARN] Video {vid_name} has only {len(frames)} frames, need at least {num_frames}. Skipping.")
        return []

    os.makedirs(output_dir, exist_ok=True)
    clips = []
    for start in range(0, max(1, len(frames) - num_frames + 1), stride):
        end = start + num_frames
        if end <= len(frames):
            clip = np.stack(frames[start:end])
        else:
            tail = end - len(frames)
            pad = np.repeat(frames[-1][None], tail, axis=0)
            clip = np.concatenate([np.stack(frames[start:]), pad], axis=0)

        if clip.shape != (num_frames, resize[1], resize[0], 3):
            warnings.warn(f"[WARN] Invalid clip shape {clip.shape} for {vid_name}. Skipping.")
            continue

        fname = f"{vid_name}_clip_{start:04d}_{end:04d}.npy"
        try:
            np.save(os.path.join(output_dir, fname), clip)
            clips.append(fname)
        except Exception as e:
            warnings.warn(f"[WARN] Failed to save clip {fname}: {e}")
            continue

        flow_clip = []
        motion_magnitudes = []
        prev_gray = cv2.cvtColor(clip[0], cv2.COLOR_BGR2GRAY)
        for i in range(1, clip.shape[0]):
            gray = cv2.cvtColor(clip[i], cv2.COLOR_BGR2GRAY)
            try:
                flow_dense = cv2.calcOpticalFlowFarneback(prev_gray, gray, None,
                                                          0.5, 3, 15, 3, 5, 1.2, 0)
                flow_dense = flow_dense.astype(np.float32)
                flow_clip.append(flow_dense)
                magnitude = np.sqrt(flow_dense[..., 0]**2 + flow_dense[..., 1]**2)
                motion_magnitudes.append(float(np.mean(magnitude)))
            except Exception as e:
                warnings.warn(f"[WARN] Flow calculation failed for {vid_name} frame {i}: {e}")
                zero_flow = np.zeros((resize[1], resize[0], 2), dtype=np.float32)
                flow_clip.append(zero_flow)
                motion_magnitudes.append(0.0)
            prev_gray = gray

        avg_motion = float(np.mean(motion_magnitudes)) if motion_magnitudes else 0.0

        if len(flow_clip) < num_frames:
            first = np.zeros((resize[1], resize[0], 2), dtype=np.float32)
            flow_clip = [first] + flow_clip

        while len(flow_clip) < num_frames:
            flow_clip.append(np.zeros((resize[1], resize[0], 2), dtype=np.float32))
        flow_clip = flow_clip[:num_frames]

        try:
            flow_array = np.stack(flow_clip)
            flow_fname = f"{vid_name}_clip_{start:04d}_{end:04d}_flow.npy"
            motion_fname = f"{vid_name}_clip_{start:04d}_{end:04d}_motion.npy"
            np.save(os.path.join(output_dir, flow_fname), flow_array)
            np.save(os.path.join(output_dir, motion_fname), np.array([avg_motion], dtype=np.float32))
        except Exception as e:
            warnings.warn(f"[WARN] Failed to save flow/motion for {fname}: {e}")

    print(f"Extracted {len(clips)} clips (+ flow + motion) from {vid_name}")
    return clips

## 4. Motion-Enhanced Label Refinement

In [ ]:
def enhanced_refine_label_with_motion(clip, cid, motion_file=None):
    diffs = [np.mean(np.abs(clip[i].astype(np.float32) - clip[i-1].astype(np.float32)))
             for i in range(1, len(clip))]
    if len(diffs) == 0:
        return cid

    avg_motion = float(np.mean(diffs))
    std_motion = float(np.std(diffs))
    max_motion = float(np.max(diffs))
    external_motion = 0.0
    if motion_file and os.path.exists(motion_file):
        try:
            external_motion = float(np.load(motion_file)[0])
        except:
            pass

    motion_score = avg_motion + 0.5 * std_motion + 0.3 * max_motion + 0.2 * external_motion

    if motion_score > 35 and cid == 0 and avg_motion > 25:
        return 1
    elif motion_score < 5 and cid == 1:
        return 0
    elif avg_motion > 30 and std_motion > 20:
        return 1

    return cid

## 5. Data Augmentation for Minority Class

In [ ]:
def create_augmentation_transforms():
    return A.ReplayCompose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.7),
        A.GaussianBlur(blur_limit=(1, 3), p=0.3),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),
    ])


def _replay_has_horizontal_flip(replay_dict):
    try:
        for t in replay_dict.get('transforms', []):
            if t.get('applied', False) and 'HorizontalFlip' in t.get('__class_fullname__', ''):
                return True
    except Exception:
        pass
    return False


def augment_clip(clip, flow, augmentation_id):
    transform = create_augmentation_transforms()
    first = clip[0].astype(np.uint8)
    result0 = transform(image=first)
    replay = result0['replay']
    flipped = _replay_has_horizontal_flip(replay)

    augmented_clip = []
    augmented_flow = []
    for i in range(clip.shape[0]):
        frame = clip[i].astype(np.uint8)
        aug = transform.replay(replay, image=frame)
        augmented_frame = aug['image']
        augmented_clip.append(augmented_frame)

        if i < flow.shape[0]:
            flow_frame = flow[i].copy()
            if flipped:
                flow_frame[..., 0] *= -1.0
            augmented_flow.append(flow_frame)
        else:
            augmented_flow.append(np.zeros_like(flow[0]))

    return np.stack(augmented_clip), np.stack(augmented_flow)


def generate_augmented_clips(clip_dir, file2label, target_ratio=0.4, num_augmentations=5):
    crossing_files = [f for f, (_, label) in file2label.items() if label == 1]
    non_crossing_files = [f for f, (_, label) in file2label.items() if label == 0]

    print(f"Original distribution: {len(crossing_files)} crossing, {len(non_crossing_files)} non-crossing")
    target_crossing_count = int(len(non_crossing_files) * target_ratio)
    augmentations_needed = max(0, target_crossing_count - len(crossing_files))
    if augmentations_needed == 0:
        print("No augmentation needed - classes already balanced")
        return file2label

    print(f"Generating {augmentations_needed} augmented crossing clips...")
    new_file2label = file2label.copy()
    augmented_count = 0

    while augmented_count < augmentations_needed:
        for original_file in crossing_files:
            if augmented_count >= augmentations_needed:
                break
            if original_file.endswith('_flow.npy') or original_file.endswith('_motion.npy'):
                continue
            try:
                clip_path = os.path.join(clip_dir, original_file)
                flow_path = os.path.join(clip_dir, original_file.replace('.npy', '_flow.npy'))
                clip = np.load(clip_path)
                if clip.ndim != 4 or clip.shape[0] == 0:
                    print(f"Skipping {original_file} - invalid shape: {clip.shape}")
                    continue
                if os.path.exists(flow_path):
                    flow = np.load(flow_path)
                    if flow.ndim != 4:
                        print(f"Warning: Invalid flow shape for {original_file}, creating zeros")
                        flow = np.zeros((clip.shape[0], clip.shape[1], clip.shape[2], 2), dtype=np.float32)
                else:
                    flow = np.zeros((clip.shape[0], clip.shape[1], clip.shape[2], 2), dtype=np.float32)

                for aug_id in range(num_augmentations):
                    if augmented_count >= augmentations_needed:
                        break
                    aug_clip, aug_flow = augment_clip(clip, flow, aug_id)
                    base_name = original_file.replace('.npy', f'_aug{aug_id}')
                    aug_clip_name = f"{base_name}.npy"
                    aug_flow_name = f"{base_name}_flow.npy"
                    np.save(os.path.join(clip_dir, aug_clip_name), aug_clip)
                    np.save(os.path.join(clip_dir, aug_flow_name), aug_flow)
                    original_text, original_label = file2label[original_file]
                    new_file2label[aug_clip_name] = (original_text, original_label)
                    augmented_count += 1
            except Exception as e:
                print(f"Warning: Failed to augment {original_file}: {e}")
                continue

    crossing_files_new = [f for f, (_, label) in new_file2label.items() if label == 1]
    non_crossing_files_new = [f for f, (_, label) in new_file2label.items() if label == 0]
    print(f"After augmentation: {len(crossing_files_new)} crossing, {len(non_crossing_files_new)} non-crossing")
    return new_file2label

## 6. Dataset Class

In [ ]:
class EnhancedPedestrianDataset(Dataset):
    def __init__(self, clip_dir, tokenizer, file_list, file2label, num_frames=30,
                 dynamic_prompt_p=0.4, is_training=True):
        self.clip_dir = clip_dir
        self.tokenizer = tokenizer
        self.files = file_list
        self.file2label = file2label
        self.num_frames = num_frames
        self.dynamic_prompt_p = dynamic_prompt_p
        self.is_training = is_training

        self.prompt_variants = {
            0: [
                "pedestrian not crossing the road",
                "person walking on sidewalk",
                "pedestrian waiting at intersection",
                "person standing near road without crossing"
            ],
            1: [
                "pedestrian crossing the road",
                "person walking across street",
                "pedestrian actively crossing intersection",
                "person traversing the roadway"
            ]
        }

    def __len__(self):
        return len(self.files)

    def get_dynamic_prompt(self, label):
        if random.random() < self.dynamic_prompt_p:
            return random.choice(self.prompt_variants[label])
        return self.prompt_variants[label][0]

    def __getitem__(self, idx):
        fname = self.files[idx]
        clip_path = os.path.join(self.clip_dir, fname)
        try:
            clip = np.load(clip_path)
            if clip.ndim != 4 or clip.shape[0] == 0:
                raise ValueError(f"Invalid clip shape: {clip.shape}")
        except Exception as e:
            print(f"Error loading clip {fname}: {e}")
            clip = np.zeros((self.num_frames, 224, 224, 3), dtype=np.uint8)

        flow_path = os.path.join(self.clip_dir, fname.replace('.npy', '_flow.npy'))
        try:
            if os.path.exists(flow_path):
                flow = np.load(flow_path)
                if flow.ndim != 4 or flow.shape[0] == 0:
                    raise ValueError(f"Invalid flow shape: {flow.shape}")
            else:
                flow = np.zeros((clip.shape[0], clip.shape[1], clip.shape[2], 2), dtype=np.float32)
        except Exception as e:
            print(f"Error loading flow for {fname}: {e}")
            flow = np.zeros((clip.shape[0], clip.shape[1], clip.shape[2], 2), dtype=np.float32)

        text, cid = self.file2label[fname]
        if self.is_training:
            text = self.get_dynamic_prompt(cid)

        tok = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=48, return_tensors='pt'
        )
        input_ids = tok['input_ids'].squeeze(0)
        attention_mask = tok['attention_mask'].squeeze(0)

        def smart_pad_or_sample(arr, target_frames):
            T = arr.shape[0]
            if T == target_frames:
                return arr
            elif T < target_frames:
                pad_len = target_frames - T
                if T > 3:
                    last_frames = arr[-3:]
                    repeats = (pad_len // 3) + 1
                    padding = np.tile(last_frames, (repeats, 1, 1, 1))[:pad_len]
                else:
                    padding = np.repeat(arr[-1][None], pad_len, axis=0)
                return np.concatenate([arr, padding], axis=0)
            else:
                indices = np.linspace(0, T - 1, target_frames, dtype=int)
                return arr[indices]

        clip = smart_pad_or_sample(clip, self.num_frames)
        flow = smart_pad_or_sample(flow, self.num_frames)

        frames = []
        for i, frame in enumerate(clip):
            img = Image.fromarray(frame.astype(np.uint8))
            if self.is_training and random.random() < 0.2:
                img = img.filter(ImageFilter.GaussianBlur(radius=0.5))
            frames.append(img)

        flow_tensor = torch.from_numpy(flow).permute(0, 3, 1, 2).float()
        return frames, flow_tensor, input_ids, attention_mask, int(cid), text, fname

## 7. Enhanced PedVLM Model Architecture

In [ ]:
class EnhancedPedVLM(nn.Module):
    def __init__(self, processor, embed_dim=768, num_classes=2):
        super().__init__()
        self.processor = processor

        self.clip_model = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
        self.t5_encoder = T5EncoderModel.from_pretrained("t5-small")

        for p in self.clip_model.parameters():
            p.requires_grad = False
        for p in self.t5_encoder.parameters():
            p.requires_grad = False

        self.visual_proj = nn.Sequential(
            nn.Linear(768, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )
        self.text_proj = nn.Sequential(
            nn.Linear(512, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )
        self.flow_encoder = nn.Sequential(
            nn.Conv2d(2, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((7, 7)),
            nn.Flatten(),
            nn.Linear(256 * 7 * 7, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2)
        )
        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=768, num_heads=8, dropout=0.1, batch_first=True
        )
        self.fusion = nn.Sequential(
            nn.Linear(embed_dim * 3, embed_dim * 2),
            nn.LayerNorm(embed_dim * 2),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(embed_dim * 2, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(embed_dim, embed_dim // 2),
            nn.LayerNorm(embed_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim // 2, embed_dim // 4),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(embed_dim // 4, num_classes)
        )

    def forward(self, frames, flows, input_ids, attention_mask):
        device = next(self.parameters()).device
        batch_size = len(frames)

        batch_images = [img for clip in frames for img in clip]
        pixel = self.processor(images=batch_images, return_tensors='pt')['pixel_values'].to(device)

        with torch.no_grad():
            vis_features = self.clip_model(pixel).last_hidden_state[:, 0, :]

        nf = len(frames[0])
        vis_features = vis_features.view(batch_size, nf, -1)

        attended_vis, _ = self.temporal_attention(vis_features, vis_features, vis_features)
        vis_combined = (vis_features + attended_vis) / 2
        vis_pool = vis_combined.mean(1)
        vproj = self.visual_proj(vis_pool)

        B, T, C, H, W = flows.shape
        flow_batch = flows.view(B*T, C, H, W).to(device)
        flow_feat = self.flow_encoder(flow_batch)
        flow_feat = flow_feat.view(B, T, -1)
        flow_weights = F.softmax(flow_feat.norm(dim=2), dim=1)
        flow_pooled = (flow_feat * flow_weights.unsqueeze(-1)).sum(1)

        with torch.no_grad():
            txt_features = self.t5_encoder(
                input_ids=input_ids.to(device),
                attention_mask=attention_mask.to(device)
            ).last_hidden_state[:, 0, :]
        tproj = self.text_proj(txt_features)

        fused = self.fusion(torch.cat([vproj, flow_pooled, tproj], dim=1))
        logits = self.classifier(fused)
        return logits

## 8. Training & Validation

In [ ]:
def train_enhanced_model(model, train_loader, val_loader, epochs, checkpoint_path, class_weights=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    optimizer = optim.AdamW([
        {'params': model.visual_proj.parameters(), 'lr': 1e-4},
        {'params': model.text_proj.parameters(), 'lr': 1e-4},
        {'params': model.flow_encoder.parameters(), 'lr': 2e-4},
        {'params': model.temporal_attention.parameters(), 'lr': 1e-4},
        {'params': model.fusion.parameters(), 'lr': 3e-4},
        {'params': model.classifier.parameters(), 'lr': 5e-4},
    ], weight_decay=1e-4)

    def focal_loss(inputs, targets, alpha=0.25, gamma=2.0):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = alpha * (1-pt)**gamma * ce_loss
        return focal_loss.mean()

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2
    )

    if class_weights is not None:
        class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
        cls_loss_fn = lambda x, y: F.cross_entropy(x, y, weight=class_weights)
    else:
        cls_loss_fn = focal_loss

    best_macro_f1 = 0.0
    patience_counter = 0
    training_history = {'train_loss': [], 'val_f1': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        correct_preds = 0
        total_preds = 0

        pbar = tqdm(train_loader, desc=f"Training Epoch {epoch+1}/{epochs}")
        for batch_idx, (frames, flows, ids, masks, labels, _, _) in enumerate(pbar):
            labels = labels.to(device)
            optimizer.zero_grad()
            logits = model(frames, flows, ids, masks)

            loss1 = cls_loss_fn(logits, labels)
            loss2 = focal_loss(logits, labels)
            loss = 0.7 * loss1 + 0.3 * loss2

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct_preds += (preds == labels).sum().item()
            total_preds += labels.size(0)
            pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'Acc': f'{100*correct_preds/total_preds:.2f}%'})

        avg_train_loss = total_loss / len(train_loader)
        train_acc = 100 * correct_preds / total_preds
        training_history['train_loss'].append(avg_train_loss)

        print(f"\nEpoch {epoch+1} Summary:")
        print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%")

        val_macro_f1, val_acc, val_report = validate_enhanced(model, val_loader, device)
        training_history['val_f1'].append(val_macro_f1)
        training_history['val_acc'].append(val_acc)

        print(f"Validation Acc: {val_acc:.4f} | Macro F1: {val_macro_f1:.4f}")
        print("Validation Report:")
        print(val_report)

        scheduler.step(val_macro_f1)

        if val_macro_f1 > best_macro_f1:
            best_macro_f1 = val_macro_f1
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'best_f1': best_macro_f1,
                'training_history': training_history
            }, checkpoint_path)
            patience_counter = 0
            print("  New best model saved!")
        else:
            patience_counter += 1

        if patience_counter >= 5:
            print(f"Early stopping after {epoch+1} epochs")
            break
        print("-" * 60)

    return model, training_history


def validate_enhanced(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    all_probs = []
    with torch.no_grad():
        for frames, flows, ids, masks, labels, _, _ in loader:
            labels = labels.to(device)
            logits = model(frames, flows, ids, masks)
            probs = F.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())
            all_probs.extend(probs.cpu().numpy())

    if len(y_true) == 0:
        return 0.0, 0.0, "No validation data"

    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    acc = np.mean([y_true[i] == y_pred[i] for i in range(len(y_true))])
    report = classification_report(y_true, y_pred, target_names=["Not Crossing", "Crossing"], zero_division=0)
    return float(macro_f1), float(acc), report

## 9. Evaluation & Visualization

In [ ]:
def plot_training_history(history, save_path):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].plot(history['train_loss'], 'b-', label='Training Loss')
    axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(history['val_f1'], 'g-', label='Validation F1')
    axes[1].set_title('Validation Macro F1 Score'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1 Score'); axes[1].legend(); axes[1].grid(True)
    axes[2].plot(history['val_acc'], 'r-', label='Validation Accuracy')
    axes[2].set_title('Validation Accuracy'); axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Accuracy'); axes[2].legend(); axes[2].grid(True)
    plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches='tight'); plt.close()


def save_enhanced_confusion_matrix(cm, labels, out_path, normalize=True):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=axes[0])
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True"); axes[0].set_title("Confusion Matrix (Counts)")
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100
    sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=axes[1])
    axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True"); axes[1].set_title("Confusion Matrix (Percentages)")
    plt.tight_layout(); plt.savefig(out_path, dpi=300, bbox_inches='tight'); plt.close()


def plot_class_distribution(y_true, y_pred, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    true_counts = Counter(y_true)
    axes[0].bar(['Not Crossing', 'Crossing'], [true_counts.get(0, 0), true_counts.get(1, 0)], color=['lightcoral', 'lightblue'])
    axes[0].set_title('True Label Distribution'); axes[0].set_ylabel('Count')
    pred_counts = Counter(y_pred)
    axes[1].bar(['Not Crossing', 'Crossing'], [pred_counts.get(0, 0), pred_counts.get(1, 0)], color=['lightcoral', 'lightblue'])
    axes[1].set_title('Predicted Label Distribution'); axes[1].set_ylabel('Count')
    plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches='tight'); plt.close()


def plot_precision_recall_curve(y_true, y_probs, save_path):
    if len(np.unique(y_true)) < 2:
        print("Cannot plot PR curve: only one class present")
        return
    precision, recall, thresholds = precision_recall_curve(y_true, y_probs[:, 1])
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, 'b-', linewidth=2)
    plt.fill_between(recall, precision, alpha=0.3)
    plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall Curve'); plt.grid(True, alpha=0.3)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
    optimal_idx = np.argmax(f1_scores[:-1])
    plt.plot(recall[optimal_idx], precision[optimal_idx], 'ro', markersize=8, label=f'Optimal F1 = {f1_scores[optimal_idx]:.3f}')
    plt.legend(); plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches='tight'); plt.close()


def write_enhanced_predictions_csv(rows, out_csv_path):
    header = ["clip_file", "true_label", "pred_label", "confidence"]
    with open(out_csv_path, "w", newline="") as f:
        w = csv.writer(f); w.writerow(header); w.writerows(rows)


def label_text(c): return "Crossing" if int(c)==1 else "Not Crossing"


def save_clip_video(npy_path, out_mp4_path, fps=15, overlay_text=None):
    clip = np.load(npy_path)
    H, W = clip.shape[1], clip.shape[2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    vw = cv2.VideoWriter(out_mp4_path, fourcc, fps, (W, H))
    for i in range(clip.shape[0]):
        frame = clip[i].copy()
        if overlay_text:
            cv2.rectangle(frame, (0,0), (W, 50), (0,0,0), -1)
            cv2.putText(frame, overlay_text, (10,35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2, cv2.LINE_AA)
        vw.write(frame)
    vw.release()


def write_performance_summary(y_true, y_pred, y_probs, per_class_metrics, out_path):
    with open(out_path, 'w') as f:
        f.write("ENHANCED PEDESTRIAN CROSSING DETECTION - PERFORMANCE SUMMARY\n" + "="*60 + "\n\n")
        acc = 100 * np.mean([y_true[i] == y_pred[i] for i in range(len(y_true))])
        macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        f.write(f"Overall Performance:\n- Accuracy: {acc:.2f}%\n- Macro F1: {macro_f1:.4f}\n\n")
        for class_name, metrics in per_class_metrics.items():
            if metrics:
                confidences = [m['confidence'] for m in metrics]
                correct_preds = [m for m in metrics if m['correct']]
                f.write(f"{class_name.title()} Class Analysis:\n")
                f.write(f"- Total samples: {len(metrics)}\n")
                f.write(f"- Correct predictions: {len(correct_preds)}\n")
                f.write(f"- Class accuracy: {100*len(correct_preds)/len(metrics):.2f}%\n")
                f.write(f"- Average confidence: {np.mean(confidences):.3f}\n")
                f.write(f"- Confidence std: {np.std(confidences):.3f}\n\n")
        cm = confusion_matrix(y_true, y_pred)
        f.write("Confusion Matrix:\n")
        f.write("                Predicted\n")
        f.write("              Not Cross  Crossing\n")
        f.write(f"True Not Cross    {cm[0,0]:6d}    {cm[0,1]:6d}\n")
        f.write(f"     Crossing     {cm[1,0]:6d}    {cm[1,1]:6d}\n\n")
        crossing_recall = cm[1,1] / (cm[1,1] + cm[1,0]) if (cm[1,1] + cm[1,0]) > 0 else 0
        crossing_precision = cm[1,1] / (cm[1,1] + cm[0,1]) if (cm[1,1] + cm[0,1]) > 0 else 0
        f.write("Model Insights:\n")
        f.write(f"- Crossing detection recall: {crossing_recall:.3f}\n")
        f.write(f"- Crossing detection precision: {crossing_precision:.3f}\n")
        if crossing_recall < 0.5: f.write("- Model struggles with crossing detection (low recall)\n")
        if crossing_precision < 0.5: f.write("- Model has many false positive crossings (low precision)\n")


def enhanced_evaluate_and_save(model, loader, device, clip_dir, pred_video_dir, out_dir, training_history=None):
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(pred_video_dir, exist_ok=True)
    model = model.to(device)
    model.eval()
    all_y, all_p = [], []
    all_probs = []
    pred_rows = []
    per_class_metrics = {'crossing': [], 'not_crossing': []}

    print("Running comprehensive evaluation...")
    with torch.no_grad():
        for frames, flows, ids, masks, labels, _, fnames in tqdm(loader, desc="Evaluating"):
            labels = labels.to(device)
            logits = model(frames, flows, ids, masks)
            probs = F.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)

            y_cpu = labels.cpu().tolist()
            p_cpu = preds.cpu().tolist()
            prob_cpu = probs.cpu().numpy()

            all_y.extend(y_cpu); all_p.extend(p_cpu); all_probs.extend(prob_cpu)

            for true_label, pred_label, prob in zip(y_cpu, p_cpu, prob_cpu):
                confidence = float(prob[pred_label])
                class_name = "crossing" if true_label == 1 else "not_crossing"
                per_class_metrics[class_name].append({'confidence': confidence, 'correct': true_label == pred_label})

            for fname, t, p, prob in zip(fnames, y_cpu, p_cpu, prob_cpu):
                confidence = float(prob[p])
                in_npy = os.path.join(clip_dir, fname)
                base = os.path.splitext(os.path.basename(fname))[0]
                out_mp4 = os.path.join(pred_video_dir, f"{base}_pred-{label_text(p)}{confidence:.2f}__true-{label_text(t)}.mp4")
                overlay = f"Pred: {label_text(p)} ({confidence:.2f}) | True: {label_text(t)}"
                try:
                    save_clip_video(in_npy, out_mp4, fps=15, overlay_text=overlay)
                except Exception as e:
                    warnings.warn(f"[WARN] Failed to save video for {fname}: {e}")
                pred_rows.append([fname, t, p, confidence])

    if len(all_y) == 0:
        print("[WARN] Test set empty; nothing to evaluate.")
        return

    all_probs = np.array(all_probs)
    acc = 100 * np.mean([all_y[i] == all_p[i] for i in range(len(all_y))])
    macro_f1 = f1_score(all_y, all_p, average='macro', zero_division=0)
    micro_f1 = f1_score(all_y, all_p, average='micro', zero_division=0)
    weighted_f1 = f1_score(all_y, all_p, average='weighted', zero_division=0)
    per_class_f1 = f1_score(all_y, all_p, average=None, zero_division=0)
    auc_score = None
    if len(np.unique(all_y)) > 1:
        try:
            auc_score = roc_auc_score(all_y, all_probs[:, 1])
        except:
            pass

    print("\n" + "="*60)
    print("COMPREHENSIVE EVALUATION RESULTS")
    print("="*60)
    print(f"Overall Accuracy: {acc:.2f}%")
    print(f"Macro F1 Score: {macro_f1:.4f}")
    print(f"Micro F1 Score: {micro_f1:.4f}")
    print(f"Weighted F1 Score: {weighted_f1:.4f}")
    if auc_score is not None:
        print(f"AUC Score: {auc_score:.4f}")
    print(f"\nPer-class F1 Scores:\nNot Crossing: {per_class_f1[0]:.4f}\nCrossing: {per_class_f1[1]:.4f}")
    print("\nDetailed Classification Report:")
    print(classification_report(all_y, all_p, target_names=["Not Crossing", "Crossing"], zero_division=0))
    true_counts = Counter(all_y); pred_counts = Counter(all_p)
    print(f"\nClass Distribution:\nTrue - Not Crossing: {true_counts.get(0, 0)}, Crossing: {true_counts.get(1, 0)}")
    print(f"Pred - Not Crossing: {pred_counts.get(0, 0)}, Crossing: {pred_counts.get(1, 0)}")

    cm = confusion_matrix(all_y, all_p, labels=[0, 1])
    cm_path = os.path.join(out_dir, "enhanced_confusion_matrix.png")
    save_enhanced_confusion_matrix(cm, ["Not Crossing", "Crossing"], cm_path)
    print(f"[Saved] Enhanced confusion matrix -> {cm_path}")

    dist_path = os.path.join(out_dir, "class_distribution.png")
    plot_class_distribution(all_y, all_p, dist_path)
    print(f"[Saved] Class distribution plot -> {dist_path}")

    if len(np.unique(all_y)) > 1:
        pr_path = os.path.join(out_dir, "precision_recall_curve.png")
        plot_precision_recall_curve(all_y, all_probs, pr_path)
        print(f"[Saved] Precision-Recall curve -> {pr_path}")

    if training_history:
        history_path = os.path.join(out_dir, "training_history.png")
        plot_training_history(training_history, history_path)
        print(f"[Saved] Training history -> {history_path}")

    csv_path = os.path.join(out_dir, "enhanced_predictions.csv")
    write_enhanced_predictions_csv(pred_rows, csv_path)
    print(f"[Saved] Enhanced predictions CSV -> {csv_path}")

    summary_path = os.path.join(out_dir, "performance_summary.txt")
    write_performance_summary(all_y, all_p, all_probs, per_class_metrics, summary_path)
    print(f"[Saved] Performance summary -> {summary_path}")
    print("="*60)

    return {
        'accuracy': acc,
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'weighted_f1': weighted_f1,
        'auc': auc_score,
        'per_class_f1': per_class_f1.tolist()
    }

## 10. Data Loading Utilities

In [ ]:
def load_xml_cache(video_files, train_ann_dir, test_ann_dir):
    cache = {}
    for f in video_files:
        vid = os.path.splitext(os.path.basename(f))[0]
        xml_train = os.path.join(train_ann_dir, vid + ".xml")
        xml_test = os.path.join(test_ann_dir, vid + ".xml")
        xml_path = xml_train if os.path.exists(xml_train) else (xml_test if os.path.exists(xml_test) else None)
        if xml_path:
            try:
                root = ET.parse(xml_path).getroot()
            except Exception as e:
                warnings.warn(f"[WARN] Failed to parse {xml_path}: {e}")
                root = ET.Element("root")
        else:
            root = ET.Element("root")
        cache[vid] = root
    return cache


def extract_frame_range_from_filename(clip_filename):
    try:
        m = re.match(r".*_clip_(\d+)_(\d+)(?:_.*)?\.npy$", clip_filename)
        if m:
            return int(m.group(1)), int(m.group(2))
        print(f"Warning: Could not parse frame range from {clip_filename}, using defaults")
        return 0, 30
    except Exception as e:
        print(f"Error parsing filename {clip_filename}: {e}")
        return 0, 30


def precompute_enhanced_labels(video_files, clip_dir, xml_cache):
    file2label = {}
    crossing_clips_found = 0
    total_clips_processed = 0
    print("\nPrecomputing labels with FIXED JAAD annotation parsing...")

    for fpath in tqdm(video_files, desc="Precompute enhanced labels"):
        vid = os.path.splitext(os.path.basename(fpath))[0]
        clips = [f for f in os.listdir(clip_dir)
                 if f.startswith(vid + "_clip_") and f.endswith('.npy')
                 and not f.endswith('_flow.npy') and not f.endswith('_motion.npy')]

        for clip_file in clips:
            total_clips_processed += 1
            clip_path = os.path.join(clip_dir, clip_file)
            motion_path = os.path.join(clip_dir, clip_file.replace('.npy', '_motion.npy'))

            try:
                clip = np.load(clip_path)
            except Exception as e:
                warnings.warn(f"[WARN] Could not load clip {clip_path}: {e}")
                continue

            start_frame, end_frame = extract_frame_range_from_filename(clip_file)
            fr_range = range(start_frame, end_frame)
            root = xml_cache.get(vid, ET.Element("root"))
            text, cid = parse_jaad_annotations(root, fr_range)
            cid = enhanced_refine_label_with_motion(clip, cid, motion_path)
            file2label[clip_file] = (text, int(cid))
            if cid == 1:
                crossing_clips_found += 1

    print(f"\nLabel computation complete:")
    print(f"Total clips processed: {total_clips_processed}")
    print(f"Crossing clips found: {crossing_clips_found}")
    print(f"Not-crossing clips: {total_clips_processed - crossing_clips_found}")
    if total_clips_processed > 0:
        print(f"Crossing percentage: {100*crossing_clips_found/total_clips_processed:.1f}%")
    return file2label


def compute_enhanced_class_weights(file2label, files, method='balanced'):
    labels = [file2label[f][1] for f in files]
    if len(labels) == 0:
        return [1.0, 1.0], labels
    labels_array = np.array(labels)
    if method == 'balanced':
        class_weights = compute_class_weight('balanced', classes=np.unique(labels_array), y=labels_array)
        return class_weights.tolist(), labels
    elif method == 'inverse':
        counts = Counter(labels); total = len(labels)
        weights = [total / (2 * counts.get(c, 1)) for c in range(2)]
        return weights, labels
    else:
        return [1.0, 1.0], labels


def collate_fn(batch):
    frames, flows, input_ids, attention_masks, labels, texts, fnames = zip(*batch)
    flow_batch = torch.stack(flows)
    ids_batch = torch.stack(input_ids)
    mask_batch = torch.stack(attention_masks)
    label_batch = torch.tensor(labels, dtype=torch.long)
    return list(frames), flow_batch, ids_batch, mask_batch, label_batch, texts, fnames


def create_enhanced_weighted_sampler(file2label, files):
    labels = [file2label[f][1] for f in files]
    class_counts = Counter(labels)
    weights = [(1.0 / class_counts[l]) if class_counts[l] > 0 else 1.0 for l in labels]
    return WeightedRandomSampler(weights, len(weights), replacement=True)

## 11. Main Pipeline

In [ ]:
def main_enhanced_pipeline(base_path, extract_stride=10, batch_size=8, epochs=15, val_split=0.2):
    seed_everything(42)
    print("="*60)
    print("ENHANCED PEDESTRIAN CROSSING DETECTION PIPELINE (Jupyter)")
    print("="*60)

    print("Step 1: Setting up environment...")
    paths = setup_environment_jupyter(base_path)

    print("\nStep 2: (Optional) Analyzing annotations...")
    try:
        analyze_all_annotations(paths["train_annotations"])
    except Exception as e:
        print(f"Annotation analysis skipped: {e}")

    print("\nStep 3: Extracting clips with optical flow and motion features...")
    for video_path in tqdm(paths["train_files"], desc="Training videos"):
        extract_clips(video_path, paths["clip_dir"], num_frames=30, stride=extract_stride)
    for video_path in tqdm(paths["test_files"], desc="Test videos"):
        extract_clips(video_path, paths["clip_dir"], num_frames=30, stride=extract_stride)

    print("\nStep 4: Computing labels with FIXED annotation parsing...")
    all_files = paths["train_files"] + paths["test_files"]
    xml_cache = load_xml_cache(all_files, paths["train_annotations"], paths["test_annotations"])
    all_clip_files = [f for f in os.listdir(paths["clip_dir"])
                      if f.endswith('.npy') and '_flow.npy' not in f and '_motion.npy' not in f]
    if len(all_clip_files) == 0:
        print("ERROR: No clip files found! Check clip extraction.")
        return
    file2label = precompute_enhanced_labels(all_files, paths["clip_dir"], xml_cache)
    if len(file2label) == 0:
        print("ERROR: No labels computed! Check annotation parsing.")
        return

    print("\nStep 5: Applying targeted data augmentation...")
    file2label = generate_augmented_clips(paths["clip_dir"], file2label, target_ratio=0.4, num_augmentations=3)

    print("\nStep 6: Creating enhanced datasets...")
    train_video_names = {os.path.splitext(os.path.basename(f))[0] for f in paths["train_files"]}
    test_video_names = {os.path.splitext(os.path.basename(f))[0] for f in paths["test_files"]}

    train_files = []
    test_files = []
    for clip_file in file2label.keys():
        video_name = clip_file.split('_clip_')[0]
        if video_name in train_video_names:
            train_files.append(clip_file)
        elif video_name in test_video_names:
            test_files.append(clip_file)
    print(f"Train clips: {len(train_files)}, Test clips: {len(test_files)}")

    random.shuffle(train_files)
    n_val = int(len(train_files) * val_split)
    val_files = train_files[:n_val]
    train_files = train_files[n_val:]
    print(f"Final split - Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}")

    print("\nStep 7: Initializing enhanced model...")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    tokenizer = T5Tokenizer.from_pretrained("t5-small")

    train_dataset = EnhancedPedestrianDataset(paths["clip_dir"], tokenizer, train_files, file2label,
                                              num_frames=30, dynamic_prompt_p=0.4, is_training=True)
    val_dataset = EnhancedPedestrianDataset(paths["clip_dir"], tokenizer, val_files, file2label,
                                            num_frames=30, dynamic_prompt_p=0.0, is_training=False)
    test_dataset = EnhancedPedestrianDataset(paths["clip_dir"], tokenizer, test_files, file2label,
                                             num_frames=30, dynamic_prompt_p=0.0, is_training=False)

    print("\nStep 8: Creating balanced data loaders...")
    class_weights, labels = compute_enhanced_class_weights(file2label, train_files, method='balanced')
    print(f"Class weights: {class_weights}")

    train_sampler = create_enhanced_weighted_sampler(file2label, train_files)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler,
                              collate_fn=collate_fn, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                            collate_fn=collate_fn, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                             collate_fn=collate_fn, num_workers=0, pin_memory=True)

    print("\nStep 9: Training enhanced model...")
    model = EnhancedPedVLM(processor, embed_dim=768, num_classes=2)
    checkpoint_path = os.path.join(paths["checkpoint_dir"], "enhanced_pedvlm_best.pt")

    trained_model, training_history = train_enhanced_model(
        model, train_loader, val_loader, epochs=epochs, checkpoint_path=checkpoint_path, class_weights=class_weights
    )

    print("\nStep 10: Loading best model and running comprehensive evaluation...")
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        model.load_state_dict(checkpoint['model_state_dict'])
        training_history = checkpoint.get('training_history', training_history)
        print(f"Loaded best model with F1: {checkpoint.get('best_f1', 'N/A'):.4f}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    print("\nStep 11: Running comprehensive evaluation...")
    print("Evaluating on validation set...")
    val_results = enhanced_evaluate_and_save(
        model, val_loader, device, paths["clip_dir"],
        os.path.join(paths["pred_video_dir"], "validation"),
        os.path.join(base_path, "validation_results"), training_history
    )
    print("Evaluating on test set...")
    test_results = enhanced_evaluate_and_save(
        model, test_loader, device, paths["clip_dir"],
        os.path.join(paths["pred_video_dir"], "test"),
        os.path.join(base_path, "test_results"), training_history
    )

    print("\n" + "="*60)
    print("FINAL RESULTS SUMMARY")
    print("="*60)
    if val_results:
        print("Validation Results:")
        for metric, value in val_results.items():
            if value is not None:
                print(f"  {metric}: {value if not isinstance(value, list) else [f'{v:.4f}' for v in value]}")
    if test_results:
        print("\nTest Results:")
        for metric, value in test_results.items():
            if value is not None:
                print(f"  {metric}: {value if not isinstance(value, list) else [f'{v:.4f}' for v in value]}")

    print("\nFiles saved:")
    print("- Model checkpoint:", checkpoint_path)
    print("- Validation results:", os.path.join(base_path, "validation_results"))
    print("- Test results:", os.path.join(base_path, "test_results"))
    print("- Prediction videos:", paths["pred_video_dir"])
    print("\n" + "="*60)
    print("PIPELINE COMPLETE!")
    print("="*60)

    return {
        'model': model,
        'training_history': training_history,
        'validation_results': val_results,
        'test_results': test_results,
        'paths': paths
    }

## 12. Quick Test (Smoke Test)

In [ ]:
def quick_test_pipeline(base_path):
    print("Running quick test pipeline...")
    seed_everything(42)
    paths = setup_environment_jupyter(base_path)

    train_files_subset = paths["train_files"][:3] if len(paths["train_files"]) >= 3 else paths["train_files"]
    test_files_subset = paths["test_files"][:2] if len(paths["test_files"]) >= 2 else paths["test_files"]
    print(f"Testing with {len(train_files_subset)} train videos, {len(test_files_subset)} test videos")

    for video_path in train_files_subset:
        extract_clips(video_path, paths["clip_dir"], num_frames=30, stride=15)

    xml_cache = load_xml_cache(train_files_subset, paths["train_annotations"], paths["test_annotations"])
    clip_files = [f for f in os.listdir(paths["clip_dir"])
                  if f.endswith('.npy') and not f.endswith('_flow.npy') and not f.endswith('_motion.npy')][:10]
    if len(clip_files) == 0:
        print("No clips found for testing")
        return

    file2label = precompute_enhanced_labels(train_files_subset, paths["clip_dir"], xml_cache)
    print(f"Created {len(file2label)} labeled clips")
    print("Label distribution:", Counter([label for _, label in file2label.values()]))
    print("Quick test completed successfully!")
    return file2label

## 13. Run Pipeline

In [ ]:
# Set this to your local dataset root
# Example structure:
# /your/path/JAAD_2/train/videos/*.mp4
# /your/path/JAAD_2/train/annotations/*.xml
# /your/path/JAAD_2/test/videos/*.mp4
# /your/path/JAAD_2/test/annotations/*.xml

BASE_PATH = r"C:\path\to\JAAD_2"  # TODO: change me

# Quick smoke test (optional)
# _ = quick_test_pipeline(BASE_PATH)

# Full pipeline (uncomment to run)
# results = main_enhanced_pipeline(BASE_PATH, extract_stride=10, batch_size=8, epochs=15, val_split=0.2)